# 3 - Portfolio sizing

**Stage 3 of 4.** Turns the walk-forward signal (`pred_ensemble`) from notebook 2 into traded books and evaluates them net of realistic costs:

- **Level 2** - constant gross notional (`weight = pred / sum(|pred|)`), with a rebalance-phase robustness sweep
- **Level 4** - constant-volatility targeting (10% annualised)
- **Level 5** - daily mean-variance optimisation (CVXPY): dollar-neutral, gross <= 1, +/-2%/name, turnover cap, and a per-name sigma/sqrt(ADV) liquidity term; solver failures carry the book forward
- Liquidity-aware transaction-cost sweep -> the headline net Sharpe

All statistics annualise with `PERIODS_PER_YEAR = 252 / TARGET_HORIZON` (weekly rebalance), not 252.

**Requires** `cvxpy`. **Output** &rarr; `Data/interim/portfolio_returns.parquet`, consumed by notebook **4 - summary & plots**.


In [ ]:
# --- Imports & setup -----------------------------------------
# Resolve paths from the repo root no matter where Jupyter launched.
import os
if os.path.basename(os.getcwd()) == "Notebooks":
    os.chdir("..")
import json
import warnings
import numpy as np
import pandas as pd
import cvxpy as cp

warnings.filterwarnings("ignore")
pd.set_option("display.float_format", "{:.6f}".format)


In [ ]:
# --- Load predictions + metadata -----------------------------
pred_df = pd.read_parquet("Data/interim/pred_df.parquet")
pred_df["date"] = pd.to_datetime(pred_df["date"])

with open("Data/interim/pipeline_meta.json") as f:
    meta = json.load(f)
TARGET_HORIZON   = meta["TARGET_HORIZON"]
REBALANCE_EVERY  = meta["REBALANCE_EVERY"]
PERIODS_PER_YEAR = meta["PERIODS_PER_YEAR"]

def sharpe(daily_returns, ann_factor=252):
    """Annualised Sharpe ratio from a per-period return series."""
    mu, sig = daily_returns.mean(), daily_returns.std()
    return 0.0 if sig < 1e-10 else mu / sig * np.sqrt(ann_factor)

print(f"Loaded pred_df {pred_df.shape}  |  {PERIODS_PER_YEAR:.1f} periods/yr")


In [ ]:
# =============================================================
# Cell 27b - Signal factor diagnostics + optional neutralization
# =============================================================
# Is the traded signal alpha, or just factor beta? We regress it
# cross-sectionally on style factors (long-horizon momentum, short-
# term reversal, low-vol, size, quality, leverage) and report its
# mean exposure to each, plus its IC, before and (optionally) after
# stripping those factors out.
#
# NEUTRALIZE_SIGNAL = False keeps the raw signal as traded (default,
# reproduces the headline). Set True to trade the factor-NEUTRAL
# residual instead - the honest test of idiosyncratic skill. (In this
# universe/period neutralising collapses IC ~0.0095 -> ~0.0029 and
# net Sharpe to ~0, i.e. the edge is mostly factor exposure.)
#
# No market-beta or sector columns exist in this panel, so those are
# omitted; dollar-neutrality already removes the market level.
# =============================================================
NEUTRALIZE_SIGNAL = False

_FSRC = ["mom_252_lag1", "mom_5_lag1", "vol_20_lag1", "roe_lag1", "debt_to_equity_lag1"]
_md = pd.read_parquet("Data/interim/model_df.parquet", columns=["date", "sec_id"] + _FSRC)
_md["date"] = pd.to_datetime(_md["date"])
_md = _md.drop_duplicates(["date", "sec_id"])
pred_df = pred_df.merge(_md, on=["date", "sec_id"], how="left")
pred_df["size_ln"] = np.log(pred_df["tc_adv"].clip(lower=1e4))
_FACTORS = ["mom_252_lag1", "mom_5_lag1", "vol_20_lag1", "roe_lag1", "debt_to_equity_lag1", "size_ln"]

_R = []
for _c in _FACTORS:
    _r = _c + "_r"; _R.append(_r)
    pred_df[_r] = pred_df.groupby("date")[_c].transform(lambda x: x.rank(pct=True) - 0.5).fillna(0.0)

def _residualize(g):
    y = g["pred_ensemble"].values.astype(float); X = g[_R].values.astype(float)
    if len(g) <= len(_R) + 2:
        return pd.Series(y - y.mean(), index=g.index)
    X = np.column_stack([np.ones(len(g)), X]); b, *_ = np.linalg.lstsq(X, y, rcond=None)
    return pd.Series(y - X @ b, index=g.index)
pred_df["pred_neut"] = pred_df.groupby("date", group_keys=False).apply(_residualize)

def _expo(sig):
    return {c: pred_df.dropna(subset=[sig, c]).groupby("date")
              .apply(lambda g: g[sig].corr(g[c], method="spearman")).mean() for c in _FACTORS}
def _sig_ic(sig):
    return pred_df.dropna(subset=[sig, "target"]).groupby("date").apply(
        lambda g: g[sig].corr(g["target"], method="spearman")).mean()

_before = _expo("pred_ensemble"); _after = _expo("pred_neut")
print("Cell 27b: Signal factor exposures (mean per-date Spearman corr)")
print(f"  {'factor':<20}{'raw':>10}{'neutralized':>14}")
for _c in _FACTORS:
    print(f"  {_c:<20}{_before[_c]:>+10.4f}{_after[_c]:>+14.4f}")
print(f"\nCell 27b: Mean IC   raw={_sig_ic('pred_ensemble'):+.5f}   neutralized={_sig_ic('pred_neut'):+.5f}")

import json as _json
_json.dump({"before": {k: float(v) for k, v in _before.items()},
            "after":  {k: float(v) for k, v in _after.items()}},
           open("Data/interim/factor_exposures.json", "w"), indent=2)
print("Cell 27b: saved -> Data/interim/factor_exposures.json")

if NEUTRALIZE_SIGNAL:
    pred_df["pred_ensemble"] = pred_df["pred_neut"]
    print("Cell 27b: NEUTRALIZE_SIGNAL=True -> trading the factor-neutral residual.")
else:
    print("Cell 27b: NEUTRALIZE_SIGNAL=False -> trading the raw signal (headline).")


In [ ]:
# =============================================================
# Cell 28 — Portfolio Construction Level 2: Constant Gross Notional
# =============================================================
# Simplest valid long-short portfolio construction.
#
# LEVEL 2 FORMULA:
# weight_i = pred_ensemble_i / sum(|pred_ensemble_j|) across all stocks
# on each REBALANCE date. pred_ensemble is the pre-registered
# walk-forward ElasticNet signal (Cell 25).
#
# v19 REBALANCING & ANNUALISATION:
# The book is formed every REBALANCE_EVERY (=TARGET_HORIZON) trading
# days and earns the H-day period return fwd_ret_h — signal horizon and
# holding period now match. There are PERIODS_PER_YEAR (≈50)
# observations per year, and ALL annualised statistics in this and
# later cells use that factor. (The v18 run used √252 on these period
# returns, inflating every Sharpe by √5 ≈ 2.24 — the source of the
# suspicious "Sharpe ≈ 3".)
#
# This ensures:
# - Long stocks with positive predictions
# - Short stocks with negative predictions
# - Total gross exposure ≈ 1.0 each day
# - Approximately dollar-neutral
#
# PnL: weights formed at rebalance date t earn fwd_ret_h — the actual
# t→t+H return. We use the raw forward return, NOT the demeaned
# target, so the PnL reflects what a real book earns.
# =============================================================

print("="*60)
print("Cell 28: PORTFOLIO LEVEL 2 — Constant Gross Notional")
print("="*60)

df_port = pred_df.dropna(subset=["pred_ensemble","fwd_ret_h"]).copy()
rebalance_dates = np.sort(df_port["date"].unique())[::REBALANCE_EVERY]
df_port = df_port[df_port["date"].isin(rebalance_dates)].copy()
df_port = df_port.sort_values(["date","ticker"]).reset_index(drop=True)

df_port["weight_l2"] = df_port.groupby("date")["pred_ensemble"].transform(
    lambda x: x / (np.abs(x).sum() + 1e-9)
)
df_port["pnl_l2"] = df_port["weight_l2"] * df_port["fwd_ret_h"]
daily_l2 = df_port.groupby("date")["pnl_l2"].sum().sort_index()

sharpe_l2  = sharpe(daily_l2, PERIODS_PER_YEAR)   # v19: ~50 periods/yr, not 252
equity_l2  = (1 + daily_l2).cumprod()
max_dd_l2  = (equity_l2 / equity_l2.cummax() - 1).min()

df_port_s = df_port.sort_values(["sec_id","date"])   # v17: entity key
df_port_s["w_prev"] = df_port_s.groupby("sec_id")["weight_l2"].shift(1).fillna(0)
df_port_s["turnover"] = (df_port_s["weight_l2"] - df_port_s["w_prev"]).abs()
avg_turnover_l2 = df_port_s.groupby("date")["turnover"].sum().mean()

print(f"Cell 28: Sharpe (annualised, {PERIODS_PER_YEAR:.1f} periods/yr): {sharpe_l2:.4f}")
print(f"Cell 28: Mean period ({TARGET_HORIZON}d) return: {daily_l2.mean():.6f}")
print(f"Cell 28: Volatility (per period):    {daily_l2.std():.6f}")
print(f"Cell 28: Max drawdown:        {max_dd_l2:.4f}")
print(f"Cell 28: Total return:        {equity_l2.iloc[-1]-1:.4f}")
print(f"Cell 28: Rebalances:          {len(daily_l2)}")
print(f"Cell 28: Avg turnover per rebalance: {avg_turnover_l2:.4f}")

# ── v19: phase robustness ──
# Sampling every H-th day admits H different rebalance grids; the
# headline should not be an artifact of one arbitrary phase. Report all.
print(f"\nCell 28: Phase robustness (Sharpe by rebalance-grid offset):")
_all_d = np.sort(pred_df.dropna(subset=["pred_ensemble", "fwd_ret_h"])["date"].unique())
_phase_sharpes = []
for _ph in range(REBALANCE_EVERY):
    _sub = pred_df[pred_df["date"].isin(_all_d[_ph::REBALANCE_EVERY])].dropna(
        subset=["pred_ensemble", "fwd_ret_h"]).copy()
    _w = _sub.groupby("date")["pred_ensemble"].transform(
        lambda x: x / (np.abs(x).sum() + 1e-9))
    _p = (_w * _sub["fwd_ret_h"]).groupby(_sub["date"]).sum()
    _s = sharpe(_p, PERIODS_PER_YEAR)
    _phase_sharpes.append(_s)
    print(f"  Cell 28: phase {_ph}: Sharpe {_s:+.3f}  ({len(_p)} rebalances)")
print(f"  Cell 28: phase range: [{min(_phase_sharpes):+.3f}, {max(_phase_sharpes):+.3f}]"
      f"  — report the range, not the best phase.")

In [ ]:
# =============================================================
# Cell 29 — Portfolio Construction Level 4: Constant Volatility
# =============================================================
# Scales the Level 2 portfolio to target constant annualised vol.
#
# MECHANISM:
# 1. Estimate rolling 20-day portfolio volatility (lagged by 1 day)
# 2. Scale factor = target_vol_daily / rolling_vol
# 3. Cap scaling at 3x to prevent extreme leverage
#
# TARGET: 10% annualised volatility (institutional baseline)
#
# WHY THIS HELPS:
# The strategy's raw vol varies across market regimes.
# Vol targeting ensures consistent risk-taking day-to-day,
# which smooths the equity curve and can improve Sharpe.
# =============================================================

print("="*60)
print("Cell 29: PORTFOLIO LEVEL 4 — Constant Volatility Targeting")
print("="*60)

TARGET_VOL_ANN    = 0.10
# v19: per-PERIOD vol target (series is one obs per rebalance)
TARGET_VOL_PERIOD = TARGET_VOL_ANN / np.sqrt(PERIODS_PER_YEAR)

# rolling(20) = 20 REBALANCES ≈ 100 trading days of trailing vol,
# lagged one period (no lookahead)
daily_l2_vol  = daily_l2.rolling(20, min_periods=10).std().shift(1)
scale_factor  = (TARGET_VOL_PERIOD / (daily_l2_vol + 1e-9)).clip(upper=3.0).fillna(1.0)
daily_l4      = daily_l2 * scale_factor

sharpe_l4     = sharpe(daily_l4, PERIODS_PER_YEAR)   # v19
equity_l4     = (1 + daily_l4).cumprod()
max_dd_l4     = (equity_l4 / equity_l4.cummax() - 1).min()
realised_vol  = daily_l4.std() * np.sqrt(PERIODS_PER_YEAR)

print(f"Cell 29: Target vol (annualised):   {TARGET_VOL_ANN:.2%}")
print(f"Cell 29: Realised vol (annualised): {realised_vol:.4f}")
print(f"Cell 29: Sharpe (annualised):       {sharpe_l4:.4f}")
print(f"Cell 29: Mean period ({TARGET_HORIZON}d) return:     {daily_l4.mean():.6f}")
print(f"Cell 29: Max drawdown:              {max_dd_l4:.4f}")
print(f"Cell 29: Total return:              {equity_l4.iloc[-1]-1:.4f}")
print(f"\nCell 25: Level 2 Sharpe: {sharpe_l2:.4f}  →  Level 4 Sharpe: {sharpe_l4:.4f}")

In [ ]:
# =============================================================
# Cell 30 — Portfolio Construction Level 5: Mean-Variance (CVXPY)
# =============================================================
# Daily Markowitz optimisation of the pre-registered signal.
#
# v18 CHANGES IN THIS CELL:
# (a) SOLVER FAILURES CARRY THE BOOK FORWARD. v17 silently
#     dropped failed days from the return series (147/834 days —
#     failures cluster on stressed/ill-conditioned covariance
#     days, so dropping them biases performance upward). v18
#     holds the previous day's weights on failed days, records
#     their PnL, and reports the carried-day count.
# (b) HORIZON-ALIGNED PnL: weights formed at t earn fwd_ret_1d
#     (the t→t+1 return), fixing v17's use of the t−1→t return.
# (c) μ is the pre-registered ElasticNet signal (continuous,
#     z-scored per day, clipped ±3, rescaled to ±0.5).
# Carried over from v17: dollar-neutrality, gross ≤ 1, ±2% per
# name, turnover ‖w−w_prev‖₁ ≤ 1.00/day, and the σ/√ADV
# liquidity cost term in the objective (median name = 10 bps).
#
# OBJECTIVE: Maximise μᵀw − γ·wᵀΣw − λᵀ|w − w_prev|
# RISK MODEL: 50% shrinkage toward scaled identity, 60-day
# lookback. SOLVER: ECOS with SCS fallback.
# =============================================================

print("=" * 60)
print("Cell 30: PORTFOLIO LEVEL 5 — Mean-Variance (CVXPY)")
print("=" * 60)
print("Cell 30: Running daily optimisation loop (10-20 minutes)...")

RISK_AVERSION     = 1.0
MAX_WEIGHT        = 0.02
SHRINKAGE         = 0.5
COV_LOOKBACK_DAYS = 60
MAX_TURNOVER      = 1.00
BASE_TC_BPS       = 10.0
USE_CONTINUOUS_MU = True

df_opt = pred_df.dropna(subset=["pred_ensemble", "fwd_ret_h"]).copy()
df_opt = df_opt.sort_values(["date", "sec_id"]).reset_index(drop=True)
# v19: keep the DAILY calendar for the covariance lookback; only the
# REBALANCE grid is subsampled. (In the v18 run, indexing the lookback
# on the subsampled grid silently stretched the "60-day" covariance
# window to 60 rebalances ≈ 300 trading days.)
all_daily_dates  = np.sort(df_opt["date"].unique())
unique_opt_dates = all_daily_dates[::REBALANCE_EVERY]

print(f"Cell 30: μ mode: {'CONTINUOUS z-scored signal' if USE_CONTINUOUS_MU else 'cross-sectional ranks'}")
print(f"Cell 30: Turnover cap {MAX_TURNOVER:.2f}/day | liquidity cost σ/√ADV, median name {BASE_TC_BPS:.0f} bps")
print("Cell 30: Solver failures carry the previous book forward (not dropped).")

results_l5   = []
failed_days  = 0     # solver failed AND no previous book existed → day skipped
carried_days = 0     # solver failed, previous book held for the day
prev_weights = {}    # sec_id -> weight from the previous day

for idx, date in enumerate(unique_opt_dates):
    day_df = df_opt[df_opt["date"] == date].copy()
    day_df = day_df.sort_values("sec_id").reset_index(drop=True)

    if len(day_df) < 30:
        failed_days += 1
        continue

    ids_today = day_df["sec_id"].values
    n = len(ids_today)

    # ── Expected returns μ (pre-registered signal) ──
    if USE_CONTINUOUS_MU:
        p  = day_df["pred_ensemble"]
        z  = ((p - p.mean()) / (p.std() + 1e-12)).clip(-3.0, 3.0)
        mu = (z / 3.0 * 0.5).values
    else:
        mu = (day_df["pred_ensemble"].rank() / n - 0.5).values

    # ── Covariance (shrinkage, 60-day lookback of realised returns) ──
    _pos = np.searchsorted(all_daily_dates, date)
    lookback_start = all_daily_dates[max(0, _pos - COV_LOOKBACK_DAYS)]
    hist = df_opt[
        (df_opt["date"] >= lookback_start) & (df_opt["date"] < date) &
        (df_opt["sec_id"].isin(ids_today))
    ]
    ret_matrix = (
        hist.pivot(index="date", columns="sec_id", values="ret")
        .reindex(columns=ids_today).fillna(0)
    )

    if ret_matrix.shape[0] < 10:
        sample_cov = np.eye(n) * (day_df["ret"].std() ** 2 + 1e-6)
    else:
        sample_cov = ret_matrix.cov().values
        sample_cov = (sample_cov + sample_cov.T) / 2
        min_eig = np.linalg.eigvalsh(sample_cov).min()
        if min_eig < 0:
            sample_cov += np.eye(n) * (-min_eig + 1e-6)

    diag_var = np.diag(sample_cov).mean()
    Sigma = (1 - SHRINKAGE) * sample_cov + SHRINKAGE * np.eye(n) * diag_var
    Sigma += 1e-6 * np.eye(n)

    # ── Per-name liquidity coefficients λᵢ ∝ σᵢ/√ADVᵢ ──
    sig = day_df["tc_sigma"].astype(float)
    adv = day_df["tc_adv"].astype(float).clip(lower=1e4)
    liq_raw = (sig / np.sqrt(adv)).replace([np.inf, -np.inf], np.nan)
    med_liq = liq_raw.median()
    if not np.isfinite(med_liq) or med_liq <= 0:
        lam = np.full(n, BASE_TC_BPS / 1e4)
    else:
        lam = (liq_raw.fillna(med_liq) / med_liq).values * (BASE_TC_BPS / 1e4)
    lam = np.clip(lam, 0.0, 10 * BASE_TC_BPS / 1e4)

    w_prev = np.array([prev_weights.get(s, 0.0) for s in ids_today])

    # ── Optimise ──
    w = cp.Variable(n)
    trades = w - w_prev
    objective = cp.Maximize(
        mu @ w
        - RISK_AVERSION * cp.quad_form(w, Sigma)
        - lam @ cp.abs(trades)
    )
    constraints = [
        cp.sum(w) == 0,
        cp.norm(w, 1) <= 1.0,
        w <= MAX_WEIGHT,
        w >= -MAX_WEIGHT,
        cp.norm(trades, 1) <= MAX_TURNOVER,
    ]
    prob = cp.Problem(objective, constraints)

    # v19: wider solver cascade — the v18 run failed on 29/121 days
    # (24%) with ECOS→SCS only; Clarabel handles this SOCP class more
    # robustly. Carry-forward still covers any residual failures.
    solved = False
    for _solver_name in ("CLARABEL", "ECOS", "SCS"):
        _solver = getattr(cp, _solver_name, None)
        if _solver is None:
            continue
        try:
            prob.solve(solver=_solver, verbose=False)
            if w.value is not None:
                solved = True
                break
        except Exception:
            continue

    if solved:
        w_val = w.value
    else:
        # v18: HOLD the previous book instead of dropping the day.
        # Names no longer in today's tradable set are treated as
        # closed at zero (documented simplification).
        if prev_weights:
            w_val = w_prev.copy()
            carried_days += 1
        else:
            failed_days += 1
            continue

    day_out = day_df.copy()
    day_out["weight_l5"] = w_val
    # horizon-aligned PnL: weights at rebalance t earn the t→t+H return
    day_out["pnl_l5"]    = w_val * day_df["fwd_ret_h"].values
    day_out["tc_lambda"] = lam
    results_l5.append(day_out)

    prev_weights = dict(zip(ids_today, w_val))

    if idx % 50 == 0:
        print(f"  Cell 30: {idx}/{len(unique_opt_dates)} dates"
              f" | carried: {carried_days} | skipped: {failed_days}")

if not results_l5:
    raise ValueError("Cell 30: No valid optimisation results.")

port_l5  = pd.concat(results_l5, ignore_index=True)
daily_l5 = port_l5.groupby("date")["pnl_l5"].sum().sort_index()

sharpe_l5 = sharpe(daily_l5, PERIODS_PER_YEAR)   # v19: ~50 periods/yr
equity_l5 = (1 + daily_l5).cumprod()
max_dd_l5 = (equity_l5 / equity_l5.cummax() - 1).min()

_p5 = port_l5.sort_values(["sec_id", "date"])
_p5["w_prev"]  = _p5.groupby("sec_id")["weight_l5"].shift(1).fillna(0)
_realised_turn = (_p5["weight_l5"] - _p5["w_prev"]).abs().groupby(_p5["date"]).sum()

print(f"\nCell 30: Optimisation complete.")
print(f"Cell 30: Solved: {len(daily_l5) - carried_days} | book carried forward:"
      f" {carried_days} | skipped (no book yet): {failed_days}")
print(f"\nCell 30: {'='*40}")
print("Cell 30: LEVEL 5 RESULTS")
print(f"Cell 30: {'='*40}")
print(f"Cell 30: Sharpe (annualised, {PERIODS_PER_YEAR:.1f} periods/yr): {sharpe_l5:.4f}")
print(f"Cell 30: Mean period ({TARGET_HORIZON}d) return: {daily_l5.mean():.6f}")
print(f"Cell 30: Volatility (per period): {daily_l5.std():.6f}")
print(f"Cell 30: Max drawdown:        {max_dd_l5:.4f}")
print(f"Cell 30: Total return:        {equity_l5.iloc[-1]-1:.4f}")
print(f"Cell 30: Rebalances:          {len(daily_l5)}")
print(f"Cell 30: Avg turnover per rebalance: {_realised_turn.mean():.4f}"
      f"  (cap = {MAX_TURNOVER:.2f}; binding on"
      f" {(_realised_turn > 0.98*MAX_TURNOVER).mean():.1%} of days)")

In [ ]:
# =============================================================
# Cell 31 — Transaction Cost Analysis
# =============================================================
# Cost sensitivity sweep for the Level 5 portfolio.
#
# v17 CHANGE (requested #3): the cost model is now LIQUIDITY-
# AWARE, using the same σᵢ/√ADVᵢ coefficient the optimizer used:
#
#   cost_t = Σᵢ |trade_i,t| × λᵢ,      λᵢ = (bps/1e4) × liqᵢ/median(liq)
#   liqᵢ  = tc_sigmaᵢ / √tc_advᵢ       (σ = 20d vol, ADV = 20d $volume,
#                                        both lagged 1 day — Cell 11)
#
# The sweep bps value is the cost of the MEDIAN-liquidity name;
# illiquid/high-vol names are charged proportionally more, very
# liquid mega-caps less. The old FLAT model (same bps for every
# name) is kept alongside for comparison — the gap between the
# two tells you how much of the book sits in expensive names.
#
# NOTE: still excludes short borrow cost and nonlinear (square-
# root) market impact (Level 6 would add these).
# The headline reported number is the LIQUIDITY-ADJUSTED 10 bps
# case, consistent with the λ used inside the optimizer.
# =============================================================

print("=" * 60)
print("Cell 31: TRANSACTION COST ANALYSIS (liquidity-aware)")
print("=" * 60)

port_l5_s = port_l5.sort_values(["sec_id", "date"])
port_l5_s["w_prev"] = (
    port_l5_s.groupby("sec_id")["weight_l5"].shift(1).fillna(0)
)
port_l5_s["trade"] = (port_l5_s["weight_l5"] - port_l5_s["w_prev"]).abs()

# liquidity multiplier: tc_lambda was calibrated in Cell 30 so the
# median name = BASE_TC_BPS. Divide it out to get a unit-median multiplier.
port_l5_s["liq_mult"] = port_l5_s["tc_lambda"] / (BASE_TC_BPS / 1e4)

daily_turnover_l5 = port_l5_s.groupby("date")["trade"].sum()
avg_turnover = daily_turnover_l5.mean()

# effective turnover: liquidity-weighted traded notional
port_l5_s["trade_liq"] = port_l5_s["trade"] * port_l5_s["liq_mult"]
daily_turnover_liq = port_l5_s.groupby("date")["trade_liq"].sum()

print(f"Cell 31: Average turnover per rebalance (Level 5): {avg_turnover:.4f}")
print(f"Cell 31: (~{avg_turnover * PERIODS_PER_YEAR:.1f}x annual portfolio turnover"
      f" — v19 fix: {PERIODS_PER_YEAR:.1f} rebalances/yr, not 252)")
print(f"Cell 31: Liquidity-weighted turnover per rebalance: {daily_turnover_liq.mean():.4f}")
print(f"Cell 31: (ratio to plain turnover = "
      f"{daily_turnover_liq.mean()/(avg_turnover+1e-12):.2f} — >1 means the book"
      f" trades disproportionately in illiquid/volatile names)\n")

print(f"Cell 31: {'Cost (bps, median name)':<26} {'Net Sharpe (flat)':<19}"
      f" {'Net Sharpe (liq-adj)':<21} {'Net Return (liq-adj)':<20}")
print("-" * 90)

for bps in [0, 5, 10, 20]:
    cost_rate = bps / 10_000
    # old flat model — every name costs the same
    daily_cost_flat = daily_turnover_l5 * cost_rate
    daily_net_flat  = (daily_l5 - daily_cost_flat).dropna()
    # NEW liquidity-adjusted model — σ/√ADV scaled per name
    daily_cost_liq  = daily_turnover_liq * cost_rate
    daily_net_liq   = (daily_l5 - daily_cost_liq).dropna()
    net_cum   = (1 + daily_net_liq).cumprod()
    net_total = net_cum.iloc[-1] - 1
    print(f"  Cell 31: {bps:<24} {sharpe(daily_net_flat, PERIODS_PER_YEAR):<19.4f}"
          f" {sharpe(daily_net_liq, PERIODS_PER_YEAR):<21.4f} {net_total:<20.4f}")

# headline: liquidity-adjusted, 10 bps median name
daily_net_10bps = (daily_l5 - daily_turnover_liq * 10 / 10_000).dropna()
sharpe_net      = sharpe(daily_net_10bps, PERIODS_PER_YEAR)   # v19
equity_net      = (1 + daily_net_10bps).cumprod()
max_dd_net      = (equity_net / equity_net.cummax() - 1).min()

print(f"\nCell 31: → Reported net Sharpe (liquidity-adjusted, 10 bps median"
      f" name): {sharpe_net:.4f}")

In [ ]:
# =============================================================
# Cell 31b - Non-linear (square-root) market impact   [Level 6]
# =============================================================
# The linear sigma/sqrt(ADV) term (Cells 30/31) prices spread plus a
# fixed per-name liquidity slope, but real market impact is CONCAVE in
# trade size: pushing a larger fraction of a name's ADV moves the price
# by ~sqrt(participation) (Almgren-style temporary impact), not linearly.
# The linear model therefore UNDER-charges big trades, and an honest net
# number needs the sqrt-impact term on top.
#
# Per name on each rebalance (weights are fractions of book GMV):
#     dollars_i       = |dw_i| * GMV
#     participation_i = dollars_i / ADV_i           (fraction of $ADV)
#     impact_i(frac)  = Y * sigma_i * sqrt(participation_i)   (sqrt-law)
#     cost_i(frac)    = |dw_i| * impact_i
# summed over names -> per-rebalance cost, subtracted from gross L5 PnL.
#
# Because sqrt-impact is SIZE-DEPENDENT (unlike the linear model, which is
# invariant to book size), GMV now matters: we sweep AUM to show capacity
# decay and report the headline at HEADLINE_GMV.
#
#   sigma_i = tc_sigma (per-name vol proxy, lagged)
#   ADV_i   = tc_adv   ($ volume, lagged)
#   Y       = dimensionless impact coefficient (~0.5-1.0; "trade 100% of
#             ADV => move price ~Y*sigma"). Absorbs the vol-horizon scaling.
#
# CAVEAT: this is charged EX-POST on the book the linear-cost optimiser
# produced. A cost-aware optimiser (sqrt-impact INSIDE the CVXPY objective,
# the true Level 6) would trade smaller and recover some of this, so the
# numbers below are a conservative lower bound, not the optimum.
# =============================================================

print("=" * 60)
print("Cell 31b: NON-LINEAR (SQUARE-ROOT) MARKET IMPACT  [Level 6]")
print("=" * 60)

Y_IMPACT     = 0.5            # Almgren temporary-impact coefficient
HEADLINE_GMV = 100_000_000   # $100M book for the reported net-of-impact number

p6 = port_l5.sort_values(["sec_id", "date"]).copy()
p6["w_prev"] = p6.groupby("sec_id")["weight_l5"].shift(1).fillna(0)
p6["dw"]     = (p6["weight_l5"] - p6["w_prev"]).abs()
sig6 = p6["tc_sigma"].astype(float)
adv6 = p6["tc_adv"].astype(float).clip(lower=1e4)
# fill gaps with cross-sectional medians so a NaN can't zero out a name
sig6 = sig6.fillna(sig6.median())
adv6 = adv6.fillna(adv6.median())

def impact_cost_series(gmv):
    participation = (p6["dw"] * gmv) / adv6
    impact = Y_IMPACT * sig6 * np.sqrt(participation)   # fractional price move
    c = p6["dw"] * impact                                # fractional cost per name
    return c.groupby(p6["date"]).sum().reindex(daily_l5.index).fillna(0.0)

spread = daily_turnover_liq * (BASE_TC_BPS / 1e4)        # linear liq (spread) baseline

print(f"Cell 31b: Y={Y_IMPACT}, sigma=tc_sigma, ADV=tc_adv | headline book = ${HEADLINE_GMV/1e6:.0f}M")
print("Cell 31b: capacity sweep - net Sharpe vs book size (sqrt-impact ON TOP of linear 10 bps):")
print(f"  {'Book (GMV)':>12} {'Impact/rebal':>13} {'Net Sharpe':>11} {'Net Return':>11}")
print("  " + "-" * 50)
for gmv in [10e6, 50e6, 100e6, 250e6, 500e6, 1_000e6]:
    cimp = impact_cost_series(gmv)
    net  = (daily_l5 - spread - cimp).dropna()
    eq   = (1 + net).cumprod()
    label = f"${gmv/1e6:.0f}M"
    print(f"  {label:>12} {cimp.mean()*1e4:>10.1f}bp"
          f" {sharpe(net, PERIODS_PER_YEAR):>11.4f} {eq.iloc[-1]-1:>10.2%}")

# headline: linear 10 bps spread  +  sqrt-impact at HEADLINE_GMV
daily_impact_head = impact_cost_series(HEADLINE_GMV)
daily_net_full = (daily_l5 - spread - daily_impact_head).dropna()
sharpe_net_full = sharpe(daily_net_full, PERIODS_PER_YEAR)
equity_net_full = (1 + daily_net_full).cumprod()
max_dd_net_full = (equity_net_full / equity_net_full.cummax() - 1).min()

print(f"\nCell 31b: -> Net Sharpe incl. sqrt-impact @ ${HEADLINE_GMV/1e6:.0f}M book: {sharpe_net_full:.4f}"
      f"  (was {sharpe_net:.4f} linear-only)")
print(f"Cell 31b:    net max drawdown: {max_dd_net_full:.4f}"
      f" | net total return: {equity_net_full.iloc[-1]-1:.4f}")


In [ ]:
# =============================================================
# Cell 31c - Level 6: sqrt-impact INSIDE the optimiser objective
# =============================================================
# Level 5 prices only linear cost, so Cell 31b's sqrt-impact was
# charged EX-POST. Level 6 puts the concave impact term in the
# objective, so the optimiser trades SMALLER in high-impact names
# and recovers part of the drag. Penalty per name:
#     kappa_i * |dw_i|^1.5,   kappa_i = Y * sigma_i * sqrt(GMV / ADV_i)
# (identical coefficient to Cell 31b, now optimised against). GMV
# enters the optimisation, so the book is tuned to its own size.
#
# NOTE: the |dw|^1.5 power cone is harder than the L5 SOCP - Clarabel
# may print "solution may be inaccurate" on some days; solutions are
# still usable (carry-forward covers any true failure).
# =============================================================
print("=" * 60)
print("Cell 31c: LEVEL 6 - sqrt-impact in the objective  (10-20 min)")
print("=" * 60)

L6_GMV = HEADLINE_GMV
L6_Y   = Y_IMPACT
results_l6, carried6, prev6 = [], 0, {}

for _idx, date in enumerate(unique_opt_dates):
    day = df_opt[df_opt["date"] == date].sort_values("sec_id").reset_index(drop=True)
    if len(day) < 30:
        continue
    ids = day["sec_id"].values; n = len(ids)
    p = day["pred_ensemble"]; z = ((p - p.mean()) / (p.std() + 1e-12)).clip(-3, 3); mu = (z / 3 * 0.5).values
    _pos = np.searchsorted(all_daily_dates, date); _lb = all_daily_dates[max(0, _pos - COV_LOOKBACK_DAYS)]
    hist = df_opt[(df_opt["date"] >= _lb) & (df_opt["date"] < date) & (df_opt["sec_id"].isin(ids))]
    rm = hist.pivot(index="date", columns="sec_id", values="ret").reindex(columns=ids).fillna(0)
    if rm.shape[0] < 10:
        cov = np.eye(n) * (day["ret"].std() ** 2 + 1e-6)
    else:
        cov = rm.cov().values; cov = (cov + cov.T) / 2
        _me = np.linalg.eigvalsh(cov).min()
        if _me < 0:
            cov += np.eye(n) * (-_me + 1e-6)
    _dv = np.diag(cov).mean(); Sig = (1 - SHRINKAGE) * cov + SHRINKAGE * np.eye(n) * _dv + 1e-6 * np.eye(n)
    sig = day["tc_sigma"].astype(float).values; adv = day["tc_adv"].astype(float).clip(lower=1e4).values
    lr = pd.Series(sig / np.sqrt(adv)).replace([np.inf, -np.inf], np.nan); ml = lr.median()
    lam = (np.full(n, BASE_TC_BPS / 1e4) if (not np.isfinite(ml) or ml <= 0)
           else np.clip((lr.fillna(ml).values / ml) * (BASE_TC_BPS / 1e4), 0, 10 * BASE_TC_BPS / 1e4))
    wp = np.array([prev6.get(s, 0.0) for s in ids])
    w = cp.Variable(n); t = w - wp
    kappa = L6_Y * sig * np.sqrt(L6_GMV / adv)
    obj = cp.Maximize(mu @ w - RISK_AVERSION * cp.quad_form(w, cp.psd_wrap(Sig))
                      - lam @ cp.abs(t) - kappa @ cp.power(cp.abs(t), 1.5))
    prob = cp.Problem(obj, [cp.sum(w) == 0, cp.norm(w, 1) <= 1,
                            w <= MAX_WEIGHT, w >= -MAX_WEIGHT, cp.norm(t, 1) <= MAX_TURNOVER])
    ok = False
    for _sn in ("CLARABEL", "SCS", "ECOS"):
        _s = getattr(cp, _sn, None)
        if _s is None:
            continue
        try:
            prob.solve(solver=_s)
            if w.value is not None:
                ok = True; break
        except Exception:
            continue
    if ok:
        wv = w.value
    elif prev6:
        wv = wp.copy(); carried6 += 1
    else:
        continue
    o = day.copy(); o["weight_l6"] = wv; o["pnl_l6"] = wv * day["fwd_ret_h"].values
    results_l6.append(o); prev6 = dict(zip(ids, wv))
    if _idx % 50 == 0:
        print(f"  Cell 31c: {_idx}/{len(unique_opt_dates)} | carried {carried6}")

port_l6  = pd.concat(results_l6, ignore_index=True)
daily_l6 = port_l6.groupby("date")["pnl_l6"].sum().sort_index()

# costs on the L6 book: linear spread + sqrt-impact @ GMV (same models as Cells 31 / 31b)
_p6 = port_l6.sort_values(["sec_id", "date"]).copy()
_p6["wp"] = _p6.groupby("sec_id")["weight_l6"].shift(1).fillna(0)
_p6["dw"] = (_p6["weight_l6"] - _p6["wp"]).abs()
_sig6 = _p6["tc_sigma"].astype(float); _adv6 = _p6["tc_adv"].astype(float).clip(lower=1e4)
_lr6 = (_sig6 / np.sqrt(_adv6)).replace([np.inf, -np.inf], np.nan); _med6 = _lr6.median()
_turnliq6 = (_p6["dw"] * (_lr6.fillna(_med6) / _med6)).groupby(_p6["date"]).sum()
_cimp6 = (_p6["dw"] * (L6_Y * _sig6 * np.sqrt(_p6["dw"] * L6_GMV / _adv6))).groupby(_p6["date"]).sum()
daily_l6_net = (daily_l6 - _turnliq6 * BASE_TC_BPS / 1e4
                - _cimp6.reindex(daily_l6.index).fillna(0)).dropna()

sharpe_l6      = sharpe(daily_l6, PERIODS_PER_YEAR)
sharpe_l6_net  = sharpe(daily_l6_net, PERIODS_PER_YEAR)
print(f"\nCell 31c: L6 gross Sharpe: {sharpe_l6:.4f} | carried {carried6}"
      f" | avg turnover {_p6.groupby('date')['dw'].sum().mean():.4f}")
print(f"Cell 31c: L6 net Sharpe (spread + sqrt-impact @ ${L6_GMV/1e6:.0f}M): {sharpe_l6_net:.4f}")
print(f"Cell 31c: vs L5 ex-post sqrt-impact @ same book: {sharpe_net_full:.4f}"
      f"  (at $100M the penalty is small, so L6 vs L5 is within optimizer noise; L6's edge grows with book size)")


In [ ]:
# =============================================================
# Neutralized-signal L5 (comparison book for the report)
# =============================================================
# Same L5 mean-variance construction, run on the factor-NEUTRAL
# residual (pred_neut from Cell 27b) regardless of NEUTRALIZE_SIGNAL,
# so the summary can show raw vs neutral equity curves side by side.
# Cost model identical to Cell 31b (linear spread + sqrt-impact @ GMV).
# Set COMPUTE_NEUTRAL_BOOK=False to skip (saves one ~3-min MVO loop).
# =============================================================
COMPUTE_NEUTRAL_BOOK = True
daily_neut_net = None
if COMPUTE_NEUTRAL_BOOK and "pred_neut" in df_opt.columns:
    print("=" * 60)
    print("Neutralized-signal L5 comparison book  (10-20 min)")
    print("=" * 60)
    _res, _carr, _prev = [], 0, {}
    for _i, date in enumerate(unique_opt_dates):
        day = df_opt[df_opt["date"] == date].sort_values("sec_id").reset_index(drop=True)
        if len(day) < 30:
            continue
        ids = day["sec_id"].values; n = len(ids)
        p = day["pred_neut"]; z = ((p - p.mean()) / (p.std() + 1e-12)).clip(-3, 3); mu = (z / 3 * 0.5).values
        _pos = np.searchsorted(all_daily_dates, date); _lb = all_daily_dates[max(0, _pos - COV_LOOKBACK_DAYS)]
        hist = df_opt[(df_opt["date"] >= _lb) & (df_opt["date"] < date) & (df_opt["sec_id"].isin(ids))]
        rm = hist.pivot(index="date", columns="sec_id", values="ret").reindex(columns=ids).fillna(0)
        if rm.shape[0] < 10:
            cov = np.eye(n) * (day["ret"].std() ** 2 + 1e-6)
        else:
            cov = rm.cov().values; cov = (cov + cov.T) / 2
            _me = np.linalg.eigvalsh(cov).min()
            if _me < 0:
                cov += np.eye(n) * (-_me + 1e-6)
        _dv = np.diag(cov).mean(); Sig = (1 - SHRINKAGE) * cov + SHRINKAGE * np.eye(n) * _dv + 1e-6 * np.eye(n)
        sig = day["tc_sigma"].astype(float).values; adv = day["tc_adv"].astype(float).clip(lower=1e4).values
        lr = pd.Series(sig / np.sqrt(adv)).replace([np.inf, -np.inf], np.nan); ml = lr.median()
        lam = (np.full(n, BASE_TC_BPS / 1e4) if (not np.isfinite(ml) or ml <= 0)
               else np.clip((lr.fillna(ml).values / ml) * (BASE_TC_BPS / 1e4), 0, 10 * BASE_TC_BPS / 1e4))
        wp = np.array([_prev.get(s, 0.0) for s in ids])
        w = cp.Variable(n); t = w - wp
        obj = cp.Maximize(mu @ w - RISK_AVERSION * cp.quad_form(w, cp.psd_wrap(Sig)) - lam @ cp.abs(t))
        prob = cp.Problem(obj, [cp.sum(w) == 0, cp.norm(w, 1) <= 1,
                                w <= MAX_WEIGHT, w >= -MAX_WEIGHT, cp.norm(t, 1) <= MAX_TURNOVER])
        ok = False
        for _sn in ("CLARABEL", "ECOS", "SCS"):
            _s = getattr(cp, _sn, None)
            if _s is None:
                continue
            try:
                prob.solve(solver=_s)
                if w.value is not None:
                    ok = True; break
            except Exception:
                continue
        if ok:
            wv = w.value
        elif _prev:
            wv = wp.copy(); _carr += 1
        else:
            continue
        o = day.copy(); o["weight_n"] = wv; o["pnl_n"] = wv * day["fwd_ret_h"].values
        _res.append(o); _prev = dict(zip(ids, wv))
    port_neut = pd.concat(_res, ignore_index=True)
    daily_neut = port_neut.groupby("date")["pnl_n"].sum().sort_index()
    _pn = port_neut.sort_values(["sec_id", "date"]).copy()
    _pn["wp"] = _pn.groupby("sec_id")["weight_n"].shift(1).fillna(0)
    _pn["dw"] = (_pn["weight_n"] - _pn["wp"]).abs()
    _sn2 = _pn["tc_sigma"].astype(float); _an = _pn["tc_adv"].astype(float).clip(lower=1e4)
    _ln = (_sn2 / np.sqrt(_an)).replace([np.inf, -np.inf], np.nan); _mn = _ln.median()
    _tln = (_pn["dw"] * (_ln.fillna(_mn) / _mn)).groupby(_pn["date"]).sum()
    _cin = (_pn["dw"] * (Y_IMPACT * _sn2 * np.sqrt(_pn["dw"] * HEADLINE_GMV / _an))).groupby(_pn["date"]).sum()
    daily_neut_net = (daily_neut - _tln * BASE_TC_BPS / 1e4
                      - _cin.reindex(daily_neut.index).fillna(0)).dropna()
    print(f"Neutralized L5: gross {sharpe(daily_neut, PERIODS_PER_YEAR):.4f}"
          f" | net (+sqrt-impact @ ${HEADLINE_GMV/1e6:.0f}M) {sharpe(daily_neut_net, PERIODS_PER_YEAR):.4f}"
          f" | carried {_carr}")
else:
    print("Neutralized comparison book skipped (COMPUTE_NEUTRAL_BOOK=False).")


In [ ]:
# =============================================================
# Persist the portfolio return series for the summary notebook
# =============================================================
port_returns = pd.DataFrame({
    "l2":        daily_l2,
    "l4":        daily_l4,
    "l5":        daily_l5,
    "net_10bps": daily_net_10bps,
    "net_full":  daily_net_full,
    "l6_net":    daily_l6_net,
}).sort_index()
port_returns.index.name = "date"
if daily_neut_net is not None:
    port_returns["neut_net"] = daily_neut_net
port_returns.to_parquet("Data/interim/portfolio_returns.parquet")

print("Saved -> Data/interim/portfolio_returns.parquet", port_returns.shape)
print(f"Level 5 net Sharpe (headline, linear 10bps): {sharpe_net:.4f}")
print(f"Level 5 net Sharpe (incl. sqrt-impact @ $100M): {sharpe_net_full:.4f}")
print(f"Level 6 net Sharpe (sqrt-impact in objective):  {sharpe_l6_net:.4f}")
